# Phase 6 - Training, version 2 hyperparameters

**In:** `data/featured`  
**Out:** MLflow run tagged `version=v2` - the tuned candidate

| Step | What |
| --- | --- |
| 1 | Setup - same split, features, metrics as v1 |
| 2 | Candidate grid |
| 3 | Sweep, scored on validation |
| 4 | Refit the winner as v2 |
| 5 | v1 vs v2 |
| 6 | Visualize |

**The grid is ranked on validation (2022), never on test.** Picking
hyperparameters by test score leaks the test set and makes the Phase 7 number
meaningless. Test is scored once, for the winner only.

Expectations are modest: v1 already beat the baseline by 11.1%, and with
calendar-only features the model is estimating a conditional mean per
station-time pattern. Tuning moves that a few percent, not double.

## Step 1 - Setup

In [1]:
import os
from pathlib import Path

import duckdb
import mlflow
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ML = Path("/opt/data/ml")
FEATURED = f"read_parquet('{ML}/featured/**/*.parquet', hive_partitioning=1)"

con = duckdb.connect()
con.execute("SET enable_progress_bar_print=false")

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("bike-demand")

TRAIN_YEARS, VAL_YEARS, TEST_YEARS = (2019, 2020, 2021), (2022,), (2023,)


def load(years):
    return con.execute(
        f"SELECT * FROM {FEATURED} WHERE year IN {years} ORDER BY start_date, hour"
    ).df()


train, val, test = load(TRAIN_YEARS), load(VAL_YEARS), load(TEST_YEARS)

assert train["start_date"].max() < val["start_date"].min(), "train/val overlap"
assert val["start_date"].max() < test["start_date"].min(), "val/test overlap"

print(f"train {len(train):>9,} | val {len(val):>9,} | test {len(test):>9,}")

train 1,999,104 | val   611,040 | test   665,760


In [2]:
TARGET = "trip_count"
CATEGORICAL = ["station_id", "season"]
NUMERIC = [
    "hour", "quarter", "month", "weekday", "week_of_year",
    "is_weekend", "is_holiday",
    "hour_sin", "hour_cos", "weekday_sin", "weekday_cos",
    "month_sin", "month_cos",
]
FEATURES = CATEGORICAL + NUMERIC
PEAK_HOURS = [7, 8, 9, 16, 17, 18, 19]

assert not (set(FEATURES) & {"year", "start_date", TARGET}), "split column leaked"


def score(y_true, y_pred, hours, label):
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    y_true = np.asarray(y_true, dtype=float)
    peak = np.isin(hours, PEAK_HOURS)
    return {
        "model": label,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAE_peak": mean_absolute_error(y_true[peak], y_pred[peak]),
        "MAE_offpeak": mean_absolute_error(y_true[~peak], y_pred[~peak]),
    }


def build(params):
    return Pipeline([
        ("prep", ColumnTransformer(
            [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
              ["season"])],
            remainder="passthrough",
        )),
        ("model", HistGradientBoostingRegressor(**params)),
    ])


DATA_CONTEXT = {
    "rows_train": len(train),
    "rows_val": len(val),
    "rows_test": len(test),
    "n_stations": train["station_id"].nunique(),
    "train_years": str(TRAIN_YEARS),
    "val_years": str(VAL_YEARS),
    "test_years": str(TEST_YEARS),
    "features": ",".join(FEATURES),
}

print(f"{len(FEATURES)} features -> {TARGET}")

15 features -> trip_count


In [3]:
# v1's numbers, pulled from its logged run rather than retyped.
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name("bike-demand")
v1_run = client.search_runs(
    experiment.experiment_id,
    filter_string="tags.version = 'v1'",
    order_by=["attributes.start_time DESC"],
    max_results=1,
)[0]

V1_PARAMS = {
    "loss": "poisson", "learning_rate": 0.1, "max_leaf_nodes": 31,
    "max_iter": 300, "min_samples_leaf": 20, "l2_regularization": 0.0,
    "early_stopping": False, "random_state": 0,
}
v1_val_mae = v1_run.data.metrics["val_MAE"]
v1_test_mae = v1_run.data.metrics["test_MAE"]

print(f"v1 run  {v1_run.info.run_id}")
print(f"v1 val  MAE {v1_val_mae:.4f}")
print(f"v1 test MAE {v1_test_mae:.4f}  (not used for selection)")

v1 run  36012b4709cf4cc3895848946e2bf504
v1 val  MAE 1.4081
v1 test MAE 1.7254  (not used for selection)


## Step 2 - Candidate grid

Twelve candidates around v1, varying capacity (`max_leaf_nodes`), how fast the
model commits (`learning_rate` x `max_iter`), and two forms of regularization.

`early_stopping` stays off throughout - sklearn's internal holdout is drawn at
random, which leaks across time on a chronological split.

In [4]:
BASE = {"loss": "poisson", "early_stopping": False, "random_state": 0}

GRID = [
    # capacity sweep at v1's learning rate
    {"learning_rate": 0.1, "max_leaf_nodes": 31, "max_iter": 300},   # = v1
    {"learning_rate": 0.1, "max_leaf_nodes": 63, "max_iter": 300},
    {"learning_rate": 0.1, "max_leaf_nodes": 127, "max_iter": 300},
    {"learning_rate": 0.1, "max_leaf_nodes": 255, "max_iter": 300},
    # slower learning, more trees
    {"learning_rate": 0.05, "max_leaf_nodes": 63, "max_iter": 600},
    {"learning_rate": 0.05, "max_leaf_nodes": 127, "max_iter": 600},
    {"learning_rate": 0.03, "max_leaf_nodes": 127, "max_iter": 900},
    # faster learning, fewer trees
    {"learning_rate": 0.2, "max_leaf_nodes": 63, "max_iter": 200},
    # regularized - larger leaves resist fitting noise in sparse station-hours
    {"learning_rate": 0.05, "max_leaf_nodes": 127, "max_iter": 600,
     "min_samples_leaf": 100},
    {"learning_rate": 0.05, "max_leaf_nodes": 127, "max_iter": 600,
     "min_samples_leaf": 200},
    {"learning_rate": 0.05, "max_leaf_nodes": 127, "max_iter": 600,
     "l2_regularization": 1.0},
    {"learning_rate": 0.05, "max_leaf_nodes": 255, "max_iter": 600,
     "l2_regularization": 1.0, "min_samples_leaf": 100},
]

print(f"{len(GRID)} candidates")
pd.DataFrame(GRID).fillna("-")

12 candidates


,learning_rate,max_leaf_nodes,max_iter,min_samples_leaf,l2_regularization
0,0.10,31,300,-,-
1,0.10,63,300,-,-
2,0.10,127,300,-,-
3,0.10,255,300,-,-
4,0.05,63,600,-,-
5,0.05,127,600,-,-
6,0.03,127,900,-,-
7,0.20,63,200,-,-
8,0.05,127,600,100.0,-
9,0.05,127,600,200.0,-


## Step 3 - Sweep

One nested MLflow run per candidate under a single parent, so the search reads
as one unit in the UI rather than twelve loose runs.

In [ ]:
%%time
results = []

with mlflow.start_run(run_name="v2_grid_search") as parent:
    mlflow.set_tags({"model": "hist_gbm", "version": "v2_search", **DATA_CONTEXT})
    mlflow.log_param("n_candidates", len(GRID))

    for i, overrides in enumerate(GRID):
        params = {**BASE, **overrides}
        with mlflow.start_run(run_name=f"cand_{i:02d}", nested=True):
            mlflow.log_params(params)
            model = build(params)
            model.fit(train[FEATURES], train[TARGET])

            # validation only - test stays sealed until the winner is chosen
            s = score(val[TARGET], model.predict(val[FEATURES]), val["hour"],
                      f"cand_{i:02d}")
            mlflow.log_metrics({f"val_{k}": v for k, v in s.items() if k != "model"})

        results.append({**overrides, "val_MAE": s["MAE"], "val_RMSE": s["RMSE"],
                        "val_MAE_peak": s["MAE_peak"]})
        print(f"cand_{i:02d}  val MAE {s['MAE']:.4f}")

leaderboard = pd.DataFrame(results).sort_values("val_MAE").reset_index(drop=True)
print(f"\nbest val MAE {leaderboard.loc[0, 'val_MAE']:.4f} "
      f"vs v1 {v1_val_mae:.4f}")

In [ ]:
leaderboard.fillna("-")

## Step 4 - Refit the winner as v2

The winning configuration is retrained and scored on test - the first and only
time the test set is touched in this phase.

In [ ]:
%%time
GRID_KEYS = [
    "learning_rate", "max_leaf_nodes", "max_iter",
    "min_samples_leaf", "l2_regularization",
]
best = leaderboard.loc[0, GRID_KEYS].dropna().to_dict()
V2_PARAMS = {**BASE, **{
    k: int(v) if k in ("max_leaf_nodes", "max_iter", "min_samples_leaf") else float(v)
    for k, v in best.items()
}}

v2_model = build(V2_PARAMS)
with mlflow.start_run(run_name="hist_gbm_v2") as run:
    mlflow.log_params(V2_PARAMS)
    mlflow.set_tags({"model": "hist_gbm", "version": "v2",
                     "selected_on": "val_MAE", **DATA_CONTEXT})

    v2_model.fit(train[FEATURES], train[TARGET])

    v2_rows = []
    for split_name, df in [("val", val), ("test", test)]:
        s = score(df[TARGET], v2_model.predict(df[FEATURES]), df["hour"], "hist_gbm_v2")
        mlflow.log_metrics({f"{split_name}_{k}": v for k, v in s.items() if k != "model"})
        v2_rows.append({"split": split_name, **s})

    mlflow.sklearn.log_model(v2_model, name="model")
    v2_run_id = run.info.run_id

v2_scores = pd.DataFrame(v2_rows)
print(f"run_id {v2_run_id}")
pd.Series(V2_PARAMS).to_frame("v2")

## Step 5 - v1 vs v2

In [ ]:
v2_val = v2_scores[v2_scores["split"] == "val"].iloc[0]
v2_test = v2_scores[v2_scores["split"] == "test"].iloc[0]

versus = pd.DataFrame([
    {"version": "v1", "val_MAE": v1_val_mae, "test_MAE": v1_test_mae,
     "test_RMSE": v1_run.data.metrics["test_RMSE"],
     "test_MAE_peak": v1_run.data.metrics["test_MAE_peak"]},
    {"version": "v2", "val_MAE": v2_val["MAE"], "test_MAE": v2_test["MAE"],
     "test_RMSE": v2_test["RMSE"], "test_MAE_peak": v2_test["MAE_peak"]},
]).set_index("version")

versus["val->test gap"] = (
    versus["test_MAE"] / versus["val_MAE"] - 1
).map("{:+.1%}".format)

print(versus.to_string())
print(f"\nv2 vs v1 on test: {1 - v2_test['MAE'] / v1_test_mae:+.2%} MAE")

## Step 6 - Visualize

Left: how the twelve candidates scored on validation. Right: v1 against v2 on
the test set.

In [ ]:
import matplotlib.pyplot as plt

BLUE, AQUA, MUTED_BAR = "#2a78d6", "#1baf7a", "#c3d9f3"
INK, MUTED, GRID_C, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(12, 4.6), facecolor=SURFACE, gridspec_kw={"width_ratios": [1.5, 1]}
)

# --- left: the sweep, best-first; the winner carries the identity color ---
order = leaderboard.index.tolist()
colors = [AQUA if i == 0 else MUTED_BAR for i in order]
ax1.bar(range(len(order)), leaderboard["val_MAE"], 0.7, color=colors, zorder=3)
ax1.axhline(v1_val_mae, color=BLUE, linewidth=2, zorder=4,
            label=f"v1 val MAE {v1_val_mae:.3f}")

ax1.set_xticks(range(len(order)), [f"{i+1}" for i in range(len(order))],
               fontsize=9, color=MUTED)
ax1.set_xlabel("candidate, best to worst", fontsize=9, color=MUTED)
ax1.set_ylabel("validation MAE", fontsize=9, color=MUTED)
ax1.set_title(f"Grid search - {len(GRID)} candidates on 2022",
              fontsize=11, color=INK, loc="left", pad=12)
ax1.set_ylim(leaderboard["val_MAE"].min() * 0.97,
             leaderboard["val_MAE"].max() * 1.02)
ax1.legend(frameon=False, fontsize=9, labelcolor=INK, loc="upper left")

# --- right: v1 vs v2 on test ---
metrics = ["test_MAE", "test_RMSE", "test_MAE_peak"]
labels = ["MAE", "RMSE", "MAE (peak)"]
x = np.arange(len(metrics))
width = 0.36
for i, (ver, color) in enumerate([("v1", BLUE), ("v2", AQUA)]):
    bars = ax2.bar(x + (i - 0.5) * width, [versus.loc[ver, m] for m in metrics],
                   width * 0.9, label=ver, color=color, zorder=3)
    # aqua sits below 3:1 on this surface - label every bar
    ax2.bar_label(bars, fmt="%.3f", padding=3, fontsize=8.5, color=INK)

ax2.set_xticks(x, labels, fontsize=10, color=INK)
ax2.set_title("v1 vs v2 - test 2023", fontsize=11, color=INK, loc="left", pad=12)
ax2.set_ylim(0, versus[metrics].to_numpy().max() * 1.22)
ax2.legend(frameon=False, ncols=2, fontsize=9, labelcolor=INK, loc="upper left")

for ax in (ax1, ax2):
    ax.set_facecolor(SURFACE)
    ax.yaxis.grid(True, color=GRID_C, linewidth=1, zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color("#c3c2b7")
    ax.tick_params(axis="both", length=0, colors=MUTED, labelsize=9)

fig.tight_layout()
fig.savefig("/tmp/phase6-v2.png", dpi=140, facecolor=SURFACE)
plt.show()